# ROSP-2: Delivered image quality (including atmosphere)

PTH Tier 1 metric **ROSP-2** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-2](https://rubinobs.atlassian.net/browse/ROSP-2)
- Confluence (Hub page; no metric page yet): [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Elana Urbach |
| Owner | Sandrine Thomas |
| Category | 2 - Data Production |
| Layer | 1 |

Method version: draft (not yet frozen). Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

**Method.** For every LSSTCam science visit of the selected programs, the delivered PSF FWHM is
`psf_sigma_median` × 2√(2 ln 2) × 0.2″/pixel from `cdb_lsstcam.visit1_quicklook`. It is reported two ways:

* **as delivered** (no correction), and
* **corrected to zenith and 500 nm**: FWHM × airmass$^{-0.6}$ × $(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{0.2}$, with the
  `lsst.summit.utils` effective wavelengths per band.

Everything is given per band and for all bands together: the median with the 25th and 75th percentiles over the
visits (table), night by night (median with 25-75 % whiskers) and as histograms. Visits need a PSF measurement, an
airmass and a band; the FWHM must lie in `FWHM_RANGE` and `filter_change_close_loop` visits are dropped.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch and the standard science blocks.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>; empty = all programs
use_cache = (
    True  # cache the ConsDB pull as parquet under ./cache (ignored on Times Square)
)

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights may be
# missing or stale; install/upgrade it there. Anywhere else this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
FWHM_RANGE = (0.3, 5.0)  # arcsec; visits outside are dropped as unphysical
EXCLUDE_REASONS = ("filter_change_close_loop",)
# Effective wavelengths used by lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam.
EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BANDS = ("u", "g", "r", "i", "z", "y")
BAND_COLORS = dict(
    zip(BANDS, ("#4a3aa7", "#2a78d6", "#1baf7a", "#eda100", "#eb6834", "#e34948"))
)
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "savefig.dpi": 150,
    }
)


def parse_day_obs(value) -> int:
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """Comma/space separated block names; bare numbers become BLOCK-<n>; empty means no program cut."""
    items = [v for v in re.split(r"[,\s]+", str(value or "").strip()) if v]
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def day_obs_chunks(day_obs_min, day_obs_max, chunk_days=31):
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    df = loader()
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
    return df

## ConsDB access

Same client and query shape as the ROSP-7 notebooks: `visit1` (pointing, band) joined to `visit1_quicklook`
(PSF) and `exposure` (`observation_reason`), one row per science visit, chunked by calendar month.

In [ ]:
USDF_CONSDB_URL = "http://consdb-pq.consdb:8080/consdb"
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "exp_time",
    "exp_midpt",
    "airmass",
    "zenith_distance",
    "altitude",
    "azimuth",
)
QUICKLOOK_COLUMNS = ("psf_sigma_median", "n_psf_star_total")
EXPOSURE_COLUMNS = ("observation_reason",)


class _SummitUtilsClient:
    """Adapter so lsst.summit.utils.ConsDbClient.query returns a DataFrame."""

    def __init__(self, url=USDF_CONSDB_URL):
        from lsst.summit.utils import ConsDbClient

        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(url)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    """Object with .query(sql) -> DataFrame: rubin_nights first, then summit_utils."""
    errors = []
    try:
        from rubin_nights import connections

        client = connections.get_clients()["consdb"]
        print("ConsDB client: rubin_nights")
        return client
    except Exception as exc:
        errors.append(f"rubin_nights: {exc!r}")
    try:
        client = _SummitUtilsClient()
        print("ConsDB client: lsst.summit.utils")
        return client
    except Exception as exc:
        errors.append(f"summit_utils: {exc!r}")
    raise RuntimeError("Could not build a ConsDB client:\n  " + "\n  ".join(errors))


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def build_query(
    day_obs_min, day_obs_max, programs, instrument="lsstcam", img_types=("science",)
):
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(client, day_obs_min, day_obs_max, programs, chunk_days=31):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    cols = list(VISIT_COLUMNS) + list(QUICKLOOK_COLUMNS) + list(EXPOSURE_COLUMNS)
    if not frames:
        return pd.DataFrame(columns=cols)
    out = pd.concat(frames, ignore_index=True)
    for c in out.columns:
        if (
            c
            not in (
                "band",
                "physical_filter",
                "science_program",
                "observation_reason",
                "img_type",
                "exp_midpt",
            )
            and out[c].dtype == object
        ):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.sort_values(["day_obs", "seq_num"]).reset_index(drop=True)

## Data, FWHM and selection

In [ ]:
start_dayobs, end_dayobs = parse_day_obs(start_dayobs), parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
use_cache = parse_bool(use_cache)
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)
cache_tag = f"v1_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs) or 'all'}"
print(f"dayObs {start_dayobs} to {end_dayobs}; programs {programs or 'all'}")

consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)

visits = raw.copy()
visits["band"] = visits["band"].astype("string").str.strip().str[0]
visits["fwhm"] = visits["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
wavelength = visits["band"].map(EFFECTIVE_WAVELENGTH_NM).astype(float)
visits["fwhm_corr"] = (
    visits["fwhm"]
    * visits["airmass"].astype(float) ** -0.6
    * (wavelength / 500.0) ** 0.2
)
visits["time"] = pd.to_datetime(
    visits["exp_midpt"], utc=True, errors="coerce"
).dt.tz_localize(None)

steps = [
    ("has PSF FWHM", visits["fwhm"].notna()),
    (f"FWHM in {FWHM_RANGE}", visits["fwhm"].between(*FWHM_RANGE)),
    (
        "has airmass >= 1 and a known band",
        (visits["airmass"] >= 1) & visits["band"].isin(BANDS),
    ),
    (
        f"observation_reason not in {EXCLUDE_REASONS}",
        ~visits["observation_reason"].isin(EXCLUDE_REASONS),
    ),
]
keep = pd.Series(True, index=visits.index)
rows = []
for name, mask in steps:
    mask = mask.fillna(False)
    rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
    keep &= mask
print(f"\nStart: {len(visits)} visits")
print(
    pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(index=False)
)
sel = visits[keep].copy()
print(f"\n{len(sel)} visits on {sel['day_obs'].nunique()} nights")
print(
    pd.crosstab(sel["science_program"], sel["band"], margins=True).reindex(
        columns=[*BANDS, "All"], fill_value=0
    )
)

## Results

### Aggregate table

Median FWHM with the 25th and 75th percentiles over the visits, per band and for all bands, as delivered and
corrected to zenith / 500 nm.

In [ ]:
def quartiles(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return (
        pd.Series(
            {"p25": x.quantile(0.25), "median": x.median(), "p75": x.quantile(0.75)}
        )
        if len(x)
        else pd.Series({"p25": np.nan, "median": np.nan, "p75": np.nan})
    )


def aggregate_table(df):
    groups = [("all", df)] + [
        (b, df[df["band"] == b]) for b in BANDS if (df["band"] == b).any()
    ]
    rows = []
    for name, g in groups:
        row = {
            "band": name,
            "n_visits": len(g),
            "n_nights": g["day_obs"].nunique(),
            "median_airmass": g["airmass"].median(),
        }
        for col, label in (("fwhm", "FWHM"), ("fwhm_corr", "FWHM zenith/500nm")):
            q = quartiles(g[col])
            row.update(
                {
                    f"{label} p25": q["p25"],
                    f"{label} median": q["median"],
                    f"{label} p75": q["p75"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index("band")


summary = aggregate_table(sel)
summary.round(3)

### Nightly time series

All bands together (top row), then one panel per band: the nightly median with 25-75 % whiskers, as delivered
(grey) and corrected to zenith / 500 nm (black for all bands, band colour otherwise).

In [ ]:
def nightly_quartiles(df, col):
    g = df.groupby("day_obs")[col]
    out = pd.DataFrame(
        {
            "p25": g.quantile(0.25),
            "median": g.median(),
            "p75": g.quantile(0.75),
            "n": g.size(),
        }
    )
    out["date"] = pd.to_datetime(out.index.astype(str))
    return out


def whiskers(ax, nightly, color, label, offset_days=0.0, ms=4):
    x = nightly["date"] + pd.Timedelta(days=offset_days)
    ax.errorbar(
        x,
        nightly["median"],
        yerr=np.array(
            [nightly["median"] - nightly["p25"], nightly["p75"] - nightly["median"]]
        ),
        fmt="o",
        ms=ms,
        color=color,
        ecolor=color,
        elinewidth=1.0,
        capsize=0,
        alpha=0.9,
        label=label,
    )


bands_present = [b for b in BANDS if (sel["band"] == b).any()]
panels = [("all", "all bands", INK)] + [
    (b, f"{b} band", BAND_COLORS[b]) for b in bands_present
]
fig, axes = plt.subplots(
    len(panels),
    1,
    figsize=(13, 2.6 * len(panels) + 1),
    sharex=True,
    sharey=True,
    squeeze=False,
)
for ax, (band, title, color) in zip(axes[:, 0], panels):
    sub = sel if band == "all" else sel[sel["band"] == band]
    whiskers(ax, nightly_quartiles(sub, "fwhm"), INK_MUTED, "as delivered", -0.15)
    whiskers(ax, nightly_quartiles(sub, "fwhm_corr"), color, "zenith / 500 nm", 0.15)
    ax.set_ylabel("FWHM (arcsec)")
    ax.set_title(
        f"{title}  (n = {len(sub)} visits, {sub['day_obs'].nunique()} nights)",
        loc="left",
        fontsize=11,
    )
axes[0, 0].legend(
    ncol=2,
    fontsize=9,
    title="marker: nightly median; whisker: 25-75 %",
    title_fontsize=9,
)
axes[-1, 0].set_xlabel("night (day_obs)")
fig.align_ylabels(axes[:, 0])
plt.show()

### Histograms

All bands overlaid, with the median (solid) and 25 / 75 % (dotted) of each marked; as delivered on the left,
corrected to zenith / 500 nm on the right. The legend quotes the three values.

In [ ]:
bins = np.arange(0.3, 3.0001, 0.04)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
for ax, (col, title) in zip(
    axes, (("fwhm", "as delivered"), ("fwhm_corr", "corrected to zenith / 500 nm"))
):
    ymax = 0
    for band in bands_present:
        d = sel.loc[sel["band"] == band, col].dropna()
        q = quartiles(d)
        counts, _, _ = ax.hist(
            d.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=2,
            color=BAND_COLORS[band],
            density=True,
            label=f'{band}: {q["median"]:.2f}" [{q["p25"]:.2f}, {q["p75"]:.2f}]  (n = {len(d)})',
        )
        ymax = max(ymax, counts.max())
    for k, band in enumerate(bands_present):
        q = quartiles(sel.loc[sel["band"] == band, col])
        y = ymax * (1.06 + 0.05 * k)
        ax.plot([q["p25"], q["p75"]], [y, y], color=BAND_COLORS[band], lw=1.2, ls=":")
        ax.plot([q["median"]], [y], marker="|", ms=10, mew=2, color=BAND_COLORS[band])
    q = quartiles(sel[col])
    ax.axvline(
        q["median"],
        color=INK,
        lw=1.2,
        label=f'all bands: {q["median"]:.2f}" [{q["p25"]:.2f}, {q["p75"]:.2f}]',
    )
    ax.axvline(q["p25"], color=INK, lw=0.8, ls=":")
    ax.axvline(q["p75"], color=INK, lw=0.8, ls=":")
    ax.set_ylim(0, ymax * (1.06 + 0.05 * len(bands_present) + 0.04))
    ax.set_xlim(bins[0], bins[-1])
    ax.set_xlabel("PSF FWHM (arcsec)")
    ax.set_title(title)
    ax.legend(fontsize=8.5, title="median [25 %, 75 %]", title_fontsize=9)
axes[0].set_ylabel("fraction of visits per bin")
plt.show()